# 01 - Exploração inicial dos dados (MLS Player Stats)

Neste notebook vamos apenas **olhar** para o CSV original, sem alterar nada.

O objetivo é entender:
- quantas linhas e colunas o arquivo tem;
- quais são os nomes reais das colunas (o FBref usa um cabeçalho "duplo",
  então esses nomes não são óbvios à primeira vista);
- quais tipos de dados cada coluna tem;
- onde existem valores ausentes;
- como as principais variáveis se distribuem (posição, clube, idade, minutos).

Isso é importante porque, seguindo a regra do projeto, **nunca vamos assumir
que uma coluna existe** -- vamos sempre conferir primeiro.

In [ ]:
# Ajusta o caminho para conseguirmos importar os módulos de "src"
# quando o notebook é executado de dentro da pasta "notebooks/".
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_players_data

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

## 1. Carregando o CSV original

Usamos a função `load_players_data`, que já lida com o cabeçalho duplo do FBref.

In [ ]:
df = load_players_data("../data/raw/mls_players.csv")
df.head()

## 2. Quantidade de jogadores (linhas) e colunas

In [ ]:
print(f"Quantidade de jogadores (linhas): {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

## 3. Nomes das colunas

Esses são os nomes *originais*, antes de qualquer limpeza. Note como o FBref
junta o grupo da estatística (ex: "Performance") com o nome dela (ex: "Gls").

In [ ]:
list(df.columns)

## 4. Tipos de dados de cada coluna

In [ ]:
df.info()

## 5. Valores ausentes por coluna

In [ ]:
df.isnull().sum()

## 6. Estatísticas descritivas das colunas numéricas

In [ ]:
df.describe()

## 7. Colunas numéricas vs. categóricas

Vamos separar automaticamente quais colunas o Pandas já reconhece como
numéricas e quais ficaram como texto (`object`).

In [ ]:
colunas_numericas = df.select_dtypes(include="number").columns.tolist()
colunas_categoricas = df.select_dtypes(include="object").columns.tolist()

print("Colunas numéricas:", colunas_numericas)
print()
print("Colunas categóricas/texto:", colunas_categoricas)

## 8. Possíveis colunas duplicadas

Verificamos se existe mais de uma coluna com valores idênticos (o que
indicaria redundância).

In [ ]:
colunas = df.columns
duplicadas = []
for i in range(len(colunas)):
    for j in range(i + 1, len(colunas)):
        if df[colunas[i]].equals(df[colunas[j]]):
            duplicadas.append((colunas[i], colunas[j]))

print("Pares de colunas com valores idênticos:", duplicadas if duplicadas else "Nenhum encontrado.")

## 9. Agora vamos limpar os dados

Usamos `prepare_players_data`, que renomeia as colunas para nomes mais
simples (ex: `Performance_Gls` -> `gls`), converte tipos, trata valores
ausentes de forma conservadora e cria métricas por 90 minutos.

In [ ]:
from src.data_cleaning import prepare_players_data

df_limpo = prepare_players_data(df)
df_limpo.head()

## 10. Distribuição de jogadores por posição (dados limpos)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_limpo["pos"].value_counts().plot(kind="bar", ax=ax, color="#1f6feb")
ax.set_title("Jogadores por posição")
ax.set_xlabel("Posição")
ax.set_ylabel("Quantidade de jogadores")
plt.show()

## 11. Distribuição de jogadores por clube

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df_limpo["squad"].value_counts().plot(kind="bar", ax=ax, color="#1f6feb")
ax.set_title("Jogadores por clube")
ax.set_xlabel("Clube")
ax.set_ylabel("Quantidade de jogadores")
plt.xticks(rotation=90)
plt.show()

## 12. Distribuição de idade

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df_limpo["age"].dropna(), bins=15, color="#1f6feb", ax=ax)
ax.set_title("Distribuição de idade dos jogadores")
ax.set_xlabel("Idade")
plt.show()

## 13. Distribuição de minutos jogados

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df_limpo["min"].dropna(), bins=20, color="#1f6feb", ax=ax)
ax.set_title("Distribuição de minutos jogados")
ax.set_xlabel("Minutos")
plt.show()

## 14. Principais estatísticas disponíveis

Este CSV é a tabela "Standard Stats" do FBref -- ele contém estatísticas
**básicas** (gols, assistências, cartões, minutos). Ele **não** contém
estatísticas avançadas como xG, xAG, chutes, passes progressivos, desarmes
ou interceptações. Por isso, as análises deste projeto (rankings,
clustering, similaridade) usam apenas as colunas que realmente existem:

In [ ]:
colunas_estatisticas = [
    c for c in df_limpo.columns
    if c not in ["rk", "player", "nation", "pos", "squad", "age", "born_year"]
]
colunas_estatisticas

## Conclusão

- O dataset tem **poucas centenas de jogadores** e um conjunto **enxuto** de estatísticas.
- Não há valores absurdos aparentes (idades e minutos dentro do esperado).
- As próximas etapas de análise (notebook `02_player_analysis.ipynb`) e
  clustering (`03_clustering.ipynb`) vão trabalhar em cima de
  `prepare_players_data`, que já vimos funcionando aqui.